# Clase 027 — concat, merge, join

**Parte 0** · VanderPlas cap. 3 §§ 3.7-3.8.

> 🎯 Juntar datasets sin generar duplicados ni perder filas. SQL-style joins en pandas.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd

## 1️⃣ `concat` — apilado simple

Alinea por index (`axis=0` apila filas) o por columnas (`axis=1`).

In [ ]:
ene = pd.DataFrame({'tienda': ['A','B'], 'monto': [100, 80]})
feb = pd.DataFrame({'tienda': ['A','B'], 'monto': [110, 90]})
mar = pd.DataFrame({'tienda': ['A','B'], 'monto': [105, 95]})

trim = pd.concat([ene, feb, mar], ignore_index=True)
print(trim)

# Con keys para mantener trazabilidad
trim_keys = pd.concat([ene, feb, mar], keys=['ene', 'feb', 'mar'])
print('\ncon keys (MultiIndex):')
print(trim_keys)

## 2️⃣ Los 4 tipos de join (SQL)

```
A  |  B          INNER     LEFT      RIGHT     OUTER
1  |  10         A∩B       A         B         A∪B
2  |  20         (común)   (todo A)  (todo B)  (todos, NaN donde falta)
3  |  --
--|  30
```

In [ ]:
clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3, 4],
    'nombre'    : ['Ana', 'Bob', 'Cris', 'Dan'],
})
ordenes = pd.DataFrame({
    'orden_id'  : [101, 102, 103, 104, 105],
    'cliente_id': [1, 1, 2, 5, 5],   # 5 no está en clientes; 3 y 4 no tienen orden
    'monto'     : [50, 80, 30, 40, 60],
})

print('clientes:')
print(clientes)
print('\nordenes:')
print(ordenes)

In [ ]:
# INNER join: solo clientes CON órdenes
print('--- INNER ---')
print(pd.merge(clientes, ordenes, on='cliente_id', how='inner'))

# LEFT join: TODOS los clientes, NaN si no tienen orden
print('\n--- LEFT ---')
print(pd.merge(clientes, ordenes, on='cliente_id', how='left'))

# OUTER: todos los clientes Y todas las órdenes
print('\n--- OUTER ---')
print(pd.merge(clientes, ordenes, on='cliente_id', how='outer'))

## 3️⃣ `validate` — atajo anti-bugs

Declara qué relación **esperas** (`1:1`, `1:m`, `m:1`, `m:m`); si los datos no la cumplen, pandas lanza excepción **antes** de generar duplicados.

In [ ]:
# Esperamos que cada cliente tenga muchas órdenes (1:m)
result = pd.merge(clientes, ordenes, on='cliente_id', how='left', validate='one_to_many')
print('OK (1:m válido)')

# Si esperaras 1:1 cuando realmente es 1:m, falla:
try:
    pd.merge(clientes, ordenes, on='cliente_id', validate='one_to_one')
except pd.errors.MergeError as e:
    print(f'\nMergeError correcto: {e}')

## 4️⃣ `indicator=True` — auditoría

Agrega columna `_merge` con `'left_only'`, `'right_only'`, `'both'`. Útil para entender qué pasó:

In [ ]:
audit = pd.merge(clientes, ordenes, on='cliente_id', how='outer', indicator=True)
print(audit)
print('\nResumen:')
print(audit['_merge'].value_counts())

## 5️⃣ `df.join` — atajo por index

Cuando ambos tienen el index alineado a la key del join, `df1.join(df2)` es más corto que `merge`:

In [ ]:
c = clientes.set_index('cliente_id')
o = ordenes.set_index('cliente_id')
print('join por index:')
print(c.join(o, how='left'))

## 6️⃣ `on` con columnas distintas: `left_on`/`right_on`

In [ ]:
tablaA = pd.DataFrame({'id_cliente': [1, 2], 'pais': ['ES', 'CL']})
tablaB = pd.DataFrame({'cliente_id': [1, 2], 'plan': ['pro', 'free']})

print(pd.merge(tablaA, tablaB, left_on='id_cliente', right_on='cliente_id'))

## ✅ Checklist

- [ ] Distingo `concat` (apilar) de `merge` (joinear)
- [ ] Elijo inner/left/right/outer según necesidad
- [ ] Uso `validate` para no generar duplicados ocultos
- [ ] Uso `indicator=True` para auditar el merge
- [ ] Conozco `df.join` como atajo por index

## 📝 Homework

Ver `README.md`. 4 joins con indicator, validate, join por index.

## 🔗 Referencias

- VanderPlas cap. 3 §§ 3.7-3.8
- [pandas Merge guide](https://pandas.pydata.org/docs/user_guide/merging.html)

➡️ **Siguiente:** [028 — groupby](../028-pandas-groupby-split-apply-combine/README.md)